In [9]:
import json
import os
import time
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
env_path = Path("/home/luis/Documents/FGV/Laboratory/document-graph/server/.env")

load_dotenv(dotenv_path=env_path)

MODELLM = "gpt-4.1-mini"
APIKEY = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key = APIKEY)

INPUT_PATH = Path("../../infra/json/graph/target_BELLICUMPHARMACEUTICALS_INC_05_07_2019-EX-10.1-Supply_Agreement.json")
OUTPUT_DIR = Path("../../infra/json/kg_extraction")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

KG_PARTS_PATH = OUTPUT_DIR / "bellicum_kg_parts.json"
KG_FULL_PATH = OUTPUT_DIR / "bellicum_contract_kg.json"
ERRORS_PATH = OUTPUT_DIR / "bellicum_kg_errors.json"

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

In [11]:
ALLOWED_NODE_TYPES = {
    "Clause",
    "DefinedTerm",
    "Party",
    "Obligation",
    "Right",
    "Permission",
    "Prohibition",
    "Condition",
    "Reference",
    "Value",
}

ALLOWED_EDGE_TYPES = {
    "CONTAINS",
    "DEFINES",
    "USES",
    "REFERENCES",
    "ASSIGNS_OBLIGATION_TO",
    "GRANTS_RIGHT_TO",
    "DEPENDS_ON",
    "MODIFIES_AMENDS",
    "SUPERSEDES",
}

def build_clause_input(node):
    return {
        "clause_id": node["id"],
        "paragraph_enum": node.get("paragraph_enum"),
        "text": (node.get("text") or "").strip(),
    }

In [ ]:
def make_extraction_prompt(clause):
    return f"""
You are extracting a Contract Knowledge Graph from a legal contract clause.

Return only valid JSON. Do not return markdown.

Task:
Extract entities and relations according to the ontology.

Node types:
- Clause
- DefinedTerm
- Party
- Obligation
- Right
- Permission
- Prohibition
- Condition
- Reference
- Value

Edge types:
- CONTAINS
- DEFINES
- USES
- REFERENCES
- ASSIGNS_OBLIGATION_TO
- GRANTS_RIGHT_TO
- DEPENDS_ON
- MODIFIES_AMENDS
- SUPERSEDES

Important rules:
- Do NOT detect contradictions in this step.
- Do NOT create CONTRADICTS relations.
- Extract only what is explicitly supported by the clause text.
- Every entity must include evidence_text.
- Every relation must include evidence_text.
- Create one Clause entity for the input clause.
- Connect the Clause entity to extracted legal entities using CONTAINS, DEFINES, USES, REFERENCES, etc.
- Use stable local IDs based on the clause_id.

Input clause:
{json.dumps(clause, indent=2)}

Return exactly this JSON structure:

{{
  "clause_id": "{clause["clause_id"]}",
  "entities": [
    {{
      "id": "string",
      "type": "Clause | DefinedTerm | Party | Obligation | Right | Permission | Prohibition | Condition | Reference | Value",
      "label": "string",
      "properties": {{}},
      "evidence_text": "string"
    }}
  ],
  "relations": [
    {{
      "source": "string",
      "target": "string",
      "type": "CONTAINS | DEFINES | USES | REFERENCES | ASSIGNS_OBLIGATION_TO | GRANTS_RIGHT_TO | DEPENDS_ON | MODIFIES_AMENDS | SUPERSEDES",
      "evidence_text": "string"
    }}
  ]
}}
"""

In [26]:
import tiktoken
enc = tiktoken.encoding_for_model(MODELLM)

def estimate_cost(input_tokens, output_tokens):
    input_cost = input_tokens * 0.0005 / 1000
    output_cost = output_tokens * 0.0015 / 1000
    return input_cost + output_cost


def safe_json_loads(text):
    text = text.strip()

    if text.startswith("```json"):
        text = text.replace("```json", "").replace("```", "").strip()
    elif text.startswith("```"):
        text = text.replace("```", "").strip()

    return json.loads(text)


def extract_kg_from_clause(client, clause):
    prompt = make_extraction_prompt(clause)

    input_tokens = len(enc.encode(prompt))

    response = client.responses.create(
        model=MODELLM,
        input=prompt,
        temperature=0
    )

    output_text = response.output_text
    output_tokens = len(enc.encode(output_text))

    cost = estimate_cost(input_tokens, output_tokens)

    return json.loads(output_text), input_tokens, output_tokens, cost

In [27]:
def validate_kg_part(kg_part):
    if not kg_part:
        return False

    if "entities" not in kg_part or "relations" not in kg_part:
        return False

    for ent in kg_part["entities"]:
        if ent.get("type") not in ALLOWED_NODE_TYPES:
            return False

    for rel in kg_part["relations"]:
        if rel.get("type") not in ALLOWED_EDGE_TYPES:
            return False

    return True

In [28]:
def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

In [31]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

MAX_WORKERS = 5 

kg_parts = []
errors = []

total_tokens = 0
total_cost = 0
lock = Lock()

nodes = [
    node for node in data["nodes"]
    if len((node.get("text") or "").strip()) >= 30
]

def process_node(i, node):
    clause = build_clause_input(node)

    try:
        kg_part, in_t, out_t, cost = extract_kg_from_clause(client, clause)

        valid = validate_kg_part(kg_part)

        return {
            "ok": valid,
            "index": i,
            "clause": clause,
            "kg_part": kg_part,
            "input_tokens": in_t,
            "output_tokens": out_t,
            "cost": cost,
            "error": None,
        }

    except Exception as e:
        return {
            "ok": False,
            "index": i,
            "clause": clause,
            "kg_part": None,
            "input_tokens": 0,
            "output_tokens": 0,
            "cost": 0,
            "error": str(e),
        }
    
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(process_node, i, node): (i, node)
        for i, node in enumerate(nodes)
    }

    for future in as_completed(futures):
        result = future.result()

        clause = result["clause"]
        in_t = result["input_tokens"]
        out_t = result["output_tokens"]
        cost = result["cost"]

        with lock:
            total_cost += cost
            total_tokens += in_t + out_t

            print(f"[{result['index']+1}/{len(nodes)}] {clause['clause_id']}")
            print(f"   tokens: in={in_t}, out={out_t}, total={in_t+out_t}")
            print(f"   cost: ${cost:.6f} | accumulated: ${total_cost:.4f}")

            if result["ok"]:
                kg_parts.append(result["kg_part"])
            else:
                errors.append({
                    "clause_id": clause["clause_id"],
                    "text": clause["text"],
                    "kg_part": result["kg_part"],
                    "error": result["error"],
                })

            if len(kg_parts) % 10 == 0:
                save_json(kg_parts, KG_PARTS_PATH)
                save_json(errors, ERRORS_PATH)

save_json(kg_parts, KG_PARTS_PATH)
save_json(errors, ERRORS_PATH)

print("Done.")
print("Valid KG parts:", len(kg_parts))
print("Errors:", len(errors))
print("Total tokens:", total_tokens)
print(f"Total estimated cost: ${total_cost:.4f}")

[5/361] target::BELLICUMPHARMACEUTICALS_INC_05_07_2019-EX-10.1-Supply_Agreement-p-9
   tokens: in=490, out=322, total=812
   cost: $0.000728 | accumulated: $0.0007
[4/361] target::BELLICUMPHARMACEUTICALS_INC_05_07_2019-EX-10.1-Supply_Agreement-p-5
   tokens: in=489, out=373, total=862
   cost: $0.000804 | accumulated: $0.0015
[1/361] target::BELLICUMPHARMACEUTICALS_INC_05_07_2019-EX-10.1-Supply_Agreement-p-1
   tokens: in=509, out=465, total=974
   cost: $0.000952 | accumulated: $0.0025
[3/361] target::BELLICUMPHARMACEUTICALS_INC_05_07_2019-EX-10.1-Supply_Agreement-p-4
   tokens: in=491, out=536, total=1027
   cost: $0.001050 | accumulated: $0.0035
[8/361] target::BELLICUMPHARMACEUTICALS_INC_05_07_2019-EX-10.1-Supply_Agreement-p-13
   tokens: in=521, out=421, total=942
   cost: $0.000892 | accumulated: $0.0044
[2/361] target::BELLICUMPHARMACEUTICALS_INC_05_07_2019-EX-10.1-Supply_Agreement-p-2
   tokens: in=497, out=581, total=1078
   cost: $0.001120 | accumulated: $0.0055
[7/361] targe

In [32]:
def merge_kg_parts(kg_parts):
    entities_by_id = {}
    relations = []

    for part in kg_parts:
        for ent in part["entities"]:
            entities_by_id[ent["id"]] = ent

        for rel in part["relations"]:
            relations.append(rel)

    return {
        "mode": "knowledge_graph",
        "knowledge_graph": {
            "entities": list(entities_by_id.values()),
            "relations": relations,
        }
    }


contract_kg = merge_kg_parts(kg_parts)
save_json(contract_kg, KG_FULL_PATH)

print("Total entities:", len(contract_kg["knowledge_graph"]["entities"]))
print("Total relations:", len(contract_kg["knowledge_graph"]["relations"]))

Total entities: 1864
Total relations: 2795
